# Grasping and Manipulation

Source orientation: printed pages 461-512, PDF pages 479-530. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

What can contacts prevent, permit, or command when a robot touches an object?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: contact mode, rolling, sliding, friction cone, form closure, force closure, grasp matrix. The important conversions for this notebook are:

- A contact constrains relative motion and can transmit only certain forces.
- Friction cones turn unilateral contact into geometry.
- Closure is a spanning question in wrench space.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Contact Geometry In This Chapter

A contact is easier to understand if we separate what it says about motion from what it says about force. On the motion side, a point on the robot and a point on the object must share compatible velocity in the constrained direction. With a hard, nonadhesive point contact, the normal direction prevents interpenetration, but the contact can open if the bodies separate. With rolling or sticking contact, selected tangential components are also constrained. With sliding contact, tangential velocity is allowed, but the allowed force must sit on the boundary or inside the friction cone. These cases are contact modes: local rules that decide which relative twists are admissible.

On the force side, each contact contributes a wrench to the object. In the planar examples below, a force `f = (f_x, f_y)` applied at point `p = (p_x, p_y)` becomes the object wrench `[tau_z, f_x, f_y]`, where `tau_z = p_x f_y - p_y f_x`. The grasp matrix collects these contact wrench columns. Reading its rank is not a complete force-closure test, because unilateral normal forces and friction inequalities still matter, but rank is a useful first alarm: if the columns cannot span all object wrench directions, no amount of actuator effort can directly resist the missing direction.

Friction cones make the inequality visible. A larger coefficient of friction widens the cone, and a linearized cone replaces the curved set of admissible forces with a small fan of rays. Each ray maps through the same contact-wrench calculation, producing a cone in wrench space. Force closure asks whether positive combinations of those rays can resist every small external wrench. Form closure is stricter in a different way: it asks whether contact geometry alone blocks every infinitesimal object motion, even before depending on tangential friction. Both closure ideas are geometric spanning questions, but they live on dual sides of the motion-force pairing.

The important engineering lesson is that a hand can look secure while the algebra says otherwise. Three contacts arranged symmetrically around a disk, each pushing through the center, balance planar forces but produce no torque about the center; the grasp matrix then has rank two, not three. Moving a contact, changing a normal, or adding frictional tangential components can create torque authority. That is why the lab below pairs a friction cone with a grasp-matrix image: the picture of contact directions and the numerical rank should be interpreted together, not as separate facts.

## Route Through the Notebook

1. Establish the objects and conventions needed for grasping and manipulation.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: contact kinematics, friction cones, and closure.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-12-grasping-and-manipulation/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| contact kinematics, friction cones, and closure | friction cone and grasp matrix | `artifacts/chapter-12-grasping-and-manipulation/figures/grasping-manipulation-lab.png` | how contact normals and friction span or fail to span object wrenches |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-12-grasping-and-manipulation/figures/grasping-manipulation-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For grasping and manipulation, the relevant failure modes are not side details; they are part of the concept. Singularities, chart boundaries, rank loss, time-scaling limits, contact mode changes, and wheel constraints are all examples of geometry asserting itself. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Build a planar grasp matrix and compare rank with friction-cone intuition.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, ask whether a rank should change, whether a path should lengthen, whether an eigenvalue should stay positive, whether a cone should widen, or whether a coordinate chart should approach a singular value. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- A visually symmetric grasp can still miss a wrench direction.
- Force closure depends on friction and contact placement together.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Manipulation is contact geometry plus actuation.
- Motion freedoms and force freedoms are dual views.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 12,
  "slug": "chapter-12-grasping-and-manipulation",
  "title": "Grasping and Manipulation",
  "notebook": "12-grasping-and-manipulation.ipynb",
  "printed_start": 461,
  "printed_end": 512,
  "pdf_start": 479,
  "pdf_end": 530,
  "part_slug": "part-04-control-contact-and-mobile-robots",
  "part_title": "Control, Contact, and Mobile Robots",
  "theme": "contact",
  "visual_focus": "contact kinematics, friction cones, and closure",
  "visual_kind": "friction cone and grasp matrix",
  "artifact_stem": "grasping-manipulation",
  "inspection_target": "how contact normals and friction span or fail to span object wrenches",
  "question": "What can contacts prevent, permit, or command when a robot touches an object?",
  "terms": [
    "contact mode",
    "rolling",
    "sliding",
    "friction cone",
    "form closure",
    "force closure",
    "grasp matrix"
  ],
  "translation": [
    "A contact constrains relative motion and can transmit only certain forces.",
    "Friction cones turn unilateral contact into geometry.",
    "Closure is a spanning question in wrench space."
  ],
  "lab": "Build a planar grasp matrix and compare rank with friction-cone intuition.",
  "pitfalls": [
    "A visually symmetric grasp can still miss a wrench direction.",
    "Force closure depends on friction and contact placement together."
  ],
  "takeaways": [
    "Manipulation is contact geometry plus actuation.",
    "Motion freedoms and force freedoms are dual views."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below is intentionally compact, but it is tied to contact reasoning. A hand Jacobian is the map that turns joint rates into contact-point velocities, so checking a planar arm Jacobian is a rehearsal for deciding which relative contact motions the mechanism can create. The rotation round trip guards the rigid-frame bookkeeping that appears whenever contact points are expressed in different frames. The twist/wrench power pairing is the key dual invariant: a wrench should do the same instantaneous work on a physical twist no matter which coordinate frame is used. If that pairing changes after an adjoint transform, a later grasp-matrix computation may look numerically plausible while assigning forces to the wrong frame.

In [ ]:
from utils.kinematics import planar_arm_points, planar_jacobian, manipulability_measure
from utils.lie import adjoint, se3_exp, se3_log, so3_exp, so3_log, transform_from, wrench_power

lengths = np.array([1.0, 0.75, 0.45])
theta = np.array([0.45, -0.8, 0.6])
points = planar_arm_points(lengths, theta)
J = planar_jacobian(lengths, theta)
R = so3_exp(np.array([0.2, -0.1, 0.35]))
round_trip = np.linalg.norm(so3_log(R) - np.array([0.2, -0.1, 0.35]))
T = transform_from(R, np.array([0.25, -0.1, 0.4]))
twist = np.array([0.1, -0.2, 0.3, 0.4, 0.2, -0.1])
wrench = np.array([0.3, 0.1, -0.2, 1.0, -0.4, 0.2])
power_before = wrench_power(twist, wrench)
power_after = wrench_power(adjoint(T) @ twist, np.linalg.inv(adjoint(T)).T @ wrench)
worked_example = {
    "endpoint": points[-1].round(4).tolist(),
    "jacobian_rank": int(np.linalg.matrix_rank(J)),
    "manipulability": float(manipulability_measure(J)),
    "rotation_log_round_trip": float(round_trip),
    "power_pairing_error": float(abs(power_before - power_after)),
}
worked_example

## Applied Lab

The applied lab changes behavior based on the chapter theme. For this chapter it samples a planar friction cone and builds a three-contact grasp matrix. Before running the cell, predict the outcome from the geometry: radial contact normals on a disk can push in several force directions, but because each normal line passes through the center, the torque row is deficient. The rank report should therefore warn that the normal-only grasp cannot command every planar wrench. The friction cone samples show the missing ingredient: tangential components can create torque, but only within the cone width allowed by the contact model.

Treat the summary as a diagnostic, not a verdict. A full closure check must include the sign constraints on normal force and the cone inequalities; an unconstrained matrix rank alone cannot prove force closure. Still, it catches an important failure mode quickly, and it gives the visual lab a numerical handle. If you widen the cone, move a contact off a radial line, or add more independent contact directions, the wrench span should improve in a way you can see in both the figure and the matrix.

In [ ]:
from utils.control import pd_response
from utils.dynamics import two_link_mass_matrix
from utils.grasping import friction_cone, grasp_matrix
from utils.mobile import mecanum_wheel_matrix, unicycle_rollout
from utils.planning import cubic_time_scaling, dijkstra_grid, quintic_time_scaling

theme = CHAPTER["theme"]
if theme in {"configuration", "kinematics"}:
    sweep = np.linspace(-np.pi, np.pi, 90)
    values = [manipulability_measure(planar_jacobian(np.array([1.0, 0.8]), np.array([0.25, q]))) for q in sweep]
    lab_summary = {"theme": theme, "max_manipulability": float(np.max(values)), "min_manipulability": float(np.min(values))}
elif theme in {"rigid", "appendix"}:
    angles = np.linspace(0.0, 0.95 * np.pi, 60)
    residuals = [np.linalg.norm(so3_log(so3_exp(np.array([0.0, 0.0, a]))) - np.array([0.0, 0.0, a])) for a in angles]
    lab_summary = {"theme": theme, "max_rotation_residual": float(np.max(residuals))}
elif theme == "dynamics":
    eigs = np.array([np.linalg.eigvalsh(two_link_mass_matrix(q)) for q in np.linspace(-np.pi, np.pi, 80)])
    lab_summary = {"theme": theme, "minimum_mass_eigenvalue": float(eigs.min())}
elif theme == "planning":
    t = np.linspace(0, 1, 100)
    lab_summary = {"theme": theme, "cubic_midpoint": float(cubic_time_scaling(np.array([0.5]), 1.0)[0]), "quintic_midpoint": float(quintic_time_scaling(np.array([0.5]), 1.0)[0])}
elif theme == "contact":
    cone = friction_cone(np.array([0.0, 1.0]), 0.5)
    G = grasp_matrix(np.array([[1.0, 0.0], [-0.5, 0.86], [-0.5, -0.86]]), np.array([[-1.0, 0.0], [0.5, -0.86], [0.5, 0.86]]))
    lab_summary = {"theme": theme, "cone_samples": int(len(cone)), "grasp_rank": int(np.linalg.matrix_rank(G))}
elif theme == "mobile":
    controls = np.c_[np.ones(80) * 0.5, np.ones(80) * 0.3]
    path = unicycle_rollout(controls)
    lab_summary = {"theme": theme, "wheel_map_rank": int(np.linalg.matrix_rank(mecanum_wheel_matrix())), "final_pose": path[-1].round(4).tolist()}
else:
    lab_summary = {"theme": theme}

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity